# LeetCode 1338 — Reduce Array Size to The Half bằng Python

Link đề: https://leetcode.com/problems/reduce-array-size-to-the-half/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def minSetSize(self, arr: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển bài toán "chọn tập giá trị để xoá" thành bài toán trên **tần suất**: mỗi giá trị chỉ còn là một con số "xoá được bao nhiêu phần tử".
2. Đếm tần suất bằng `dict` / `collections.Counter`, và hiểu vì sao **tham lam chọn tần suất lớn nhất trước** là tối ưu (lập luận hoán đổi).
3. Viết 4 cách: thử mọi tập con (baseline) → sort tần suất giảm dần → heap → **counting sort theo tần suất `O(n)`**.
4. Xử lý chính xác điều kiện "**ít nhất một nửa**" (`>=`, so sánh bằng số nguyên `2·removed >= n`).
5. Tránh lỗi: dùng `>` thay `>=`, sắp tần suất tăng dần, sắp theo **giá trị** thay vì tần suất, đếm bằng `arr.count` trong vòng lặp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` (độ dài **chẵn**). Bạn chọn một **tập** các số nguyên và xoá **mọi lần xuất hiện** của chúng khỏi mảng. Trả về **kích thước nhỏ nhất** của tập sao cho **ít nhất một nửa** số phần tử của mảng bị xoá.

**Ví dụ 1:** `arr = [3,3,3,3,5,5,5,2,2,7]` → `2`
(Chọn `{3,7}` ⇒ còn `[5,5,5,2,2]` dài 5 = một nửa. Các tập cỡ 2 hợp lệ khác: `{3,5}`, `{3,2}`, `{5,2}`. Tập `{2,7}` không hợp lệ vì còn 7 phần tử.)

**Ví dụ 2:** `arr = [7,7,7,7,7,7]` → `1`

**Ràng buộc:** `2 <= arr.length <= 10^5`; `arr.length` chẵn; `1 <= arr[i] <= 10^5`.

In [1]:
import heapq
import random
import time
from collections import Counter
from itertools import combinations
from typing import List

examples = [([3, 3, 3, 3, 5, 5, 5, 2, 2, 7], 2), ([7, 7, 7, 7, 7, 7], 1)]
for arr, want in examples:
    print(arr, "| n =", len(arr), "| một nửa =", len(arr) // 2, "-> mong đợi", want)

[3, 3, 3, 3, 5, 5, 5, 2, 2, 7] | n = 10 | một nửa = 5 -> mong đợi 2
[7, 7, 7, 7, 7, 7] | n = 6 | một nửa = 3 -> mong đợi 1


## 3. Kiến thức nền

### 3.1 Đếm tần suất bằng `dict`

Duyệt mảng một lần: `freq[x] = freq.get(x, 0) + 1`. Tổng thời gian `O(n)`.

In [2]:
arr = [3, 3, 3, 3, 5, 5, 5, 2, 2, 7]
freq = {}
for x in arr:
    freq[x] = freq.get(x, 0) + 1
print(freq)

{3: 4, 5: 3, 2: 2, 7: 1}


### 3.2 `collections.Counter`

`Counter(arr)` làm đúng việc trên; `.values()` là các tần suất; `.most_common()` trả các cặp `(giá trị, tần suất)` theo tần suất giảm dần.

In [3]:
cnt = Counter(arr)
print(cnt)
print("tần suất:", list(cnt.values()))
print("most_common:", cnt.most_common())

Counter({3: 4, 5: 3, 2: 2, 7: 1})
tần suất: [4, 3, 2, 1]
most_common: [(3, 4), (5, 3), (2, 2), (7, 1)]


### 3.3 Sắp giảm dần và tổng tích luỹ

`sorted(vals, reverse=True)`; tổng tích luỹ bằng `itertools.accumulate` hoặc biến cộng dồn trong vòng lặp.

In [4]:
from itertools import accumulate
vals = sorted(cnt.values(), reverse=True)
print(vals, "-> tích luỹ:", list(accumulate(vals)))

[4, 3, 2, 1] -> tích luỹ: [4, 7, 9, 10]


### 3.4 Max-heap bằng `heapq` (đổi dấu)

`heapq` chỉ có min-heap; muốn lấy **lớn nhất** trước thì đẩy `-x`. `heapify` biến list thành heap trong `O(k)`.

In [5]:
h = [-v for v in cnt.values()]
heapq.heapify(h)
print([-heapq.heappop(h) for _ in range(len(h))])

[4, 3, 2, 1]


### 3.5 So sánh "ít nhất một nửa" bằng số nguyên

"Đã xoá ít nhất một nửa" ⇔ `removed >= n / 2` ⇔ `2 * removed >= n`. Viết dạng nhân 2 tránh số thực và chắc chắn đúng cả khi `n` lẻ (dù đề cho `n` chẵn).

In [6]:
n = 10
for removed in (4, 5, 6):
    print(removed, "->", 2 * removed >= n, removed >= n // 2, removed > n // 2)

4 -> False False False
5 -> True True False
6 -> True True True


### 3.6 Liệt kê tập con với `combinations`

`combinations(items, r)` sinh mọi tập con `r` phần tử — dùng cho brute force.

In [7]:
print(list(combinations([3, 5, 2, 7], 2)))

[(3, 5), (3, 2), (3, 7), (5, 2), (5, 7), (2, 7)]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `n = 10`, cần xoá **ít nhất 5** phần tử.

| giá trị | 3 | 5 | 2 | 7 |
|---|---|---|---|---|
| tần suất | 4 | 3 | 2 | 1 |

- Chọn 1 giá trị: tốt nhất là `3` xoá được 4 < 5 ⇒ **không đủ**.
- Chọn 2 giá trị: `3` + `5` xoá 7 ≥ 5 ⇒ **đủ** ⇒ đáp án `2`.

Trực giác: mỗi giá trị chọn vào tập "tốn" 1 đơn vị kích thước, nhưng xoá được **số phần tử bằng tần suất của nó**. Muốn tốn ít nhất ⇒ chọn những giá trị **xoá được nhiều nhất** trước. Giá trị cụ thể không quan trọng, chỉ tần suất là quan trọng.

In [8]:
arr = examples[0][0]
vals = sorted(Counter(arr).values(), reverse=True)
removed = 0
for k, f in enumerate(vals, 1):
    removed += f
    print(f"chọn thêm giá trị có tần suất {f}: đã xoá {removed}/{len(arr)} -> {'ĐỦ' if 2 * removed >= len(arr) else 'chưa đủ'} với {k} giá trị")
    if 2 * removed >= len(arr):
        break

chọn thêm giá trị có tần suất 4: đã xoá 4/10 -> chưa đủ với 1 giá trị
chọn thêm giá trị có tần suất 3: đã xoá 7/10 -> ĐỦ với 2 giá trị


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** gọi `d` = số giá trị phân biệt. Thử kích thước `k = 1, 2, …`; với mỗi `k`, thử **mọi** tập `k` giá trị; nếu tổng tần suất của tập `≥ n/2` thì trả về `k`. Kích thước đầu tiên tìm được là nhỏ nhất.

In [9]:
def min_set_size_brute(arr):
    n = len(arr)
    cnt = Counter(arr)
    items = list(cnt)
    for k in range(1, len(items) + 1):
        for subset in combinations(items, k):
            if 2 * sum(cnt[x] for x in subset) >= n:
                return k
    return len(items)

for arr, want in examples:
    print(min_set_size_brute(arr), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


**Độ phức tạp:** xấu nhất thử `2^d` tập con, mỗi tập tốn `O(d)` ⇒ `O(2^d · d)`. Với `d` có thể tới `5·10^4` ⇒ hoàn toàn không khả thi; chỉ dùng cho `d ≤ 15` để kiểm chứng.

**Nút thắt:** ta thử mù quáng mọi tổ hợp, trong khi chỉ cần hỏi "với `k` giá trị thì **xoá được tối đa** bao nhiêu?" — câu trả lời rõ ràng là tổng `k` **tần suất lớn nhất**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (tham lam):** với kích thước tập cố định `k`, số phần tử xoá được **lớn nhất** là tổng `k` tần suất lớn nhất. Do đó đáp án = số `k` nhỏ nhất sao cho tổng `k` tần suất lớn nhất `≥ n/2`.

**Lập luận hoán đổi:** giả sử một tập tối ưu `S` không chứa giá trị `a` có tần suất lớn nhất, nhưng chứa `b` với `f(b) ≤ f(a)`. Thay `b` bằng `a`: kích thước không đổi, số phần tử xoá không giảm ⇒ vẫn hợp lệ. Lặp lại ⇒ tồn tại tập tối ưu gồm các tần suất lớn nhất.

**Quy tắc "nếu… thì…":**
- Sắp tần suất giảm dần, cộng dồn.
- Nếu `2 · removed >= n` thì dừng, trả số giá trị đã chọn.

**Bước nâng cao:** tần suất là số nguyên trong `[1, n]` ⇒ có thể "sắp" bằng **đếm** (bucket) trong `O(n)` thay vì `O(d log d)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` + sắp tần suất giảm dần (tham lam)

**Từng đoạn:**
1. `freqs = sorted(Counter(arr).values(), reverse=True)`.
2. Cộng dồn `removed`; mỗi lần cộng tăng `ans`.
3. Khi `2 * removed >= n` ⇒ trả `ans`.

In [10]:
def min_set_size_sort(arr):
    n = len(arr)
    removed = 0
    for ans, f in enumerate(sorted(Counter(arr).values(), reverse=True), 1):
        removed += f
        if 2 * removed >= n:
            return ans
    return 0                     # không tới được vì tổng tần suất = n

for arr, want in examples:
    print(min_set_size_sort(arr), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


Code nộp LeetCode (Cách 2):

In [11]:
class SolutionSort:
    def minSetSize(self, arr: list[int]) -> int:
        n = len(arr)
        removed = 0
        for ans, f in enumerate(sorted(Counter(arr).values(), reverse=True), 1):
            removed += f
            if 2 * removed >= n:
                return ans
        return 0

print([SolutionSort().minSetSize(a) for a, _ in examples])

[2, 1]


**Độ phức tạp:** đếm `O(n)`, sắp `O(d log d)` (`d` = số giá trị phân biệt `≤ n`) ⇒ `O(n + d log d)` thời gian, `O(d)` bộ nhớ.

### 7.2 Cách 3 — Max-heap (lấy dần tần suất lớn nhất)

Thay vì sắp toàn bộ, `heapify` các tần suất (đổi dấu) trong `O(d)`, rồi `heappop` từng cái lớn nhất cho đến khi đủ. Nếu đáp án `k` nhỏ, chỉ tốn `O(d + k log d)` — có lợi khi đáp án nhỏ so với `d`.

In [12]:
def min_set_size_heap(arr):
    n = len(arr)
    heap = [-f for f in Counter(arr).values()]
    heapq.heapify(heap)
    removed = ans = 0
    while 2 * removed < n:
        removed -= heapq.heappop(heap)
        ans += 1
    return ans

for arr, want in examples:
    print(min_set_size_heap(arr), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


Code nộp LeetCode (Cách 3):

In [13]:
class SolutionHeap:
    def minSetSize(self, arr: list[int]) -> int:
        n = len(arr)
        heap = [-f for f in Counter(arr).values()]
        heapq.heapify(heap)
        removed = ans = 0
        while 2 * removed < n:
            removed -= heapq.heappop(heap)
            ans += 1
        return ans

print([SolutionHeap().minSetSize(a) for a, _ in examples])

[2, 1]


**Độ phức tạp:** `O(n + d + k log d)` thời gian, `O(d)` bộ nhớ. (Tương tự, `heapq.nlargest` hay `Counter.most_common()` cũng dùng được.)

### 7.3 Cách 4 — Counting sort theo tần suất (`O(n)`)

**Ý tưởng:** mọi tần suất nằm trong `[1, n]`. Tạo `bucket[f]` = **số giá trị** có tần suất đúng `f`. Duyệt `f` từ `n` xuống `1`: ở mỗi `f` có `bucket[f]` giá trị như nhau; lấy **vừa đủ** số giá trị cần thiết bằng phép chia làm tròn lên, khỏi lặp từng cái.

**Từng đoạn:**
1. `cnt = Counter(arr)`; `bucket = [0] * (n + 1)`; với mỗi tần suất `f`: `bucket[f] += 1`.
2. `need = (n + 1) // 2` (số phần tử tối thiểu phải xoá).
3. Với `f = n..1`, nếu `bucket[f] > 0`: số giá trị cần lấy ở mức này `take = min(bucket[f], ceil(need / f))`; `ans += take`; `need -= take * f`; nếu `need <= 0` trả `ans`.

In [14]:
def min_set_size_bucket(arr):
    n = len(arr)
    bucket = [0] * (n + 1)
    for f in Counter(arr).values():
        bucket[f] += 1
    need = (n + 1) // 2
    ans = 0
    for f in range(n, 0, -1):
        if bucket[f] == 0:
            continue
        take = min(bucket[f], (need + f - 1) // f)      # ceil(need / f)
        ans += take
        need -= take * f
        if need <= 0:
            return ans
    return ans

for arr, want in examples:
    print(min_set_size_bucket(arr), "| mong đợi", want)

2 | mong đợi 2
1 | mong đợi 1


Code nộp LeetCode (Cách 4):

In [15]:
class SolutionBucket:
    def minSetSize(self, arr: list[int]) -> int:
        n = len(arr)
        freq = [0] * (max(arr) + 1)
        for x in arr:
            freq[x] += 1
        bucket = [0] * (n + 1)
        for f in freq:
            if f:
                bucket[f] += 1
        need, ans = (n + 1) // 2, 0
        for f in range(n, 0, -1):
            if bucket[f]:
                take = min(bucket[f], (need + f - 1) // f)
                ans += take
                need -= take * f
                if need <= 0:
                    return ans
        return ans

print([SolutionBucket().minSetSize(a) for a, _ in examples])

[2, 1]


**Độ phức tạp:** `O(n + max(arr))` thời gian (đếm bằng mảng vì `arr[i] ≤ 10^5`), `O(n + max(arr))` bộ nhớ — **tuyến tính**, không cần so sánh để sắp.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** (`n = 10`, cần `2·removed ≥ 10`):

| bước | tần suất lấy | removed | 2·removed ≥ 10? | ans |
|---|---|---|---|---|
| 1 | 4 (giá trị 3) | 4 | 8 — chưa | 1 |
| 2 | 3 (giá trị 5) | 7 | 14 — **đủ** | 2 |

In [16]:
def trace_sort(arr):
    n = len(arr)
    pairs = sorted(Counter(arr).items(), key=lambda p: -p[1])
    print(f"n={n}, (giá trị, tần suất) giảm dần: {pairs}")
    removed = 0
    for ans, (v, f) in enumerate(pairs, 1):
        removed += f
        ok = 2 * removed >= n
        print(f"  bước {ans}: lấy {v} (tần suất {f}) -> removed={removed}, 2*removed={2 * removed} >= {n}? {ok}")
        if ok:
            print("  => đáp án", ans)
            return ans

trace_sort(examples[0][0])

n=10, (giá trị, tần suất) giảm dần: [(3, 4), (5, 3), (2, 2), (7, 1)]
  bước 1: lấy 3 (tần suất 4) -> removed=4, 2*removed=8 >= 10? False
  bước 2: lấy 5 (tần suất 3) -> removed=7, 2*removed=14 >= 10? True
  => đáp án 2


2

Trace Cách 4 (bucket) trên một mảng có nhiều giá trị cùng tần suất:

In [17]:
def trace_bucket(arr):
    n = len(arr)
    bucket = [0] * (n + 1)
    for f in Counter(arr).values():
        bucket[f] += 1
    print("bucket (tần suất: số giá trị):", {f: b for f, b in enumerate(bucket) if b})
    need, ans = (n + 1) // 2, 0
    print("cần xoá ít nhất", need)
    for f in range(n, 0, -1):
        if bucket[f]:
            take = min(bucket[f], (need + f - 1) // f)
            ans += take
            need -= take * f
            print(f"  f={f}: có {bucket[f]} giá trị, lấy {take} -> ans={ans}, còn cần {max(need, 0)}")
            if need <= 0:
                return ans

arr_demo = [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 5, 6, 7, 8, 9]
print("đáp án:", trace_bucket(arr_demo))

bucket (tần suất: số giá trị): {1: 5, 2: 1, 3: 3}
cần xoá ít nhất 8
  f=3: có 3 giá trị, lấy 3 -> ans=3, còn cần 0
đáp án: 3


## 9. Code nộp LeetCode

In [18]:
class Solution:
    def minSetSize(self, arr: list[int]) -> int:
        n = len(arr)
        removed = 0
        for ans, f in enumerate(sorted(Counter(arr).values(), reverse=True), 1):
            removed += f
            if 2 * removed >= n:          # đã xoá ít nhất một nửa
                return ans
        return 0

for arr, want in examples:
    got = Solution().minSetSize(arr)
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

2 | mong đợi 2 | OK
1 | mong đợi 1 | OK


## 10. Vì sao thuật toán đúng

Gọi các tần suất sắp giảm dần là `f_1 ≥ f_2 ≥ … ≥ f_d`, và `P_k = f_1 + … + f_k`.

**(a) Mọi tập `k` giá trị xoá được không quá `P_k` phần tử:** tổng `k` số bất kỳ trong dãy không vượt quá tổng `k` số lớn nhất.

**(b) Tập gồm `k` giá trị có tần suất lớn nhất xoá đúng `P_k` phần tử.**

Gọi `k*` là chỉ số nhỏ nhất với `2·P_{k*} ≥ n` (tồn tại vì `P_d = n`). Theo (b), tập `k*` giá trị tần suất lớn nhất hợp lệ ⇒ đáp án `≤ k*`. Với mọi `k < k*`, theo (a), mọi tập cỡ `k` xoá `≤ P_k < n/2` ⇒ không hợp lệ ⇒ đáp án `≥ k*`. Vậy đáp án `= k*`, đúng giá trị thuật toán trả về.

**Cách 4:** duyệt `f` giảm dần là duyệt các tần suất theo đúng thứ tự giảm; trong một mức `f`, lấy `ceil(need / f)` giá trị là số nhỏ nhất đủ bù `need`, giống hệt việc cộng từng cái một rồi dừng.

## 11. Phân tích Time / Space

Đặt `n = len(arr)`, `d` = số giá trị phân biệt, `k` = đáp án, `M = max(arr)`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập con | `O(2^d · d)` | `O(d)` |
| 2. Counter + sort | `O(n + d log d)` | `O(d)` |
| 3. Heap | `O(n + d + k log d)` | `O(d)` |
| 4. Bucket theo tần suất | `O(n + M)` (hoặc `O(n)` với `Counter`) | `O(n + M)` |

Lưu ý: `d` có thể bằng `n` (mọi phần tử khác nhau, khi đó đáp án là `n/2`), nên không thể coi `d` là nhỏ; Cách 4 không phụ thuộc vào `d`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
solvers = {
    "brute": min_set_size_brute,
    "sort": min_set_size_sort,
    "heap": min_set_size_heap,
    "bucket": min_set_size_bucket,
    "submit": lambda a: Solution().minSetSize(a),
    "submit_sort": lambda a: SolutionSort().minSetSize(a),
    "submit_heap": lambda a: SolutionHeap().minSetSize(a),
    "submit_bucket": lambda a: SolutionBucket().minSetSize(a),
}
cases = examples + [
    ([1, 9], 1),                              # n = 2
    ([1, 2, 3, 4, 5, 6], 3),                  # mọi phần tử khác nhau -> n/2
    ([1, 1, 2, 2], 1),                        # đúng bằng một nửa
    ([2, 2, 1, 1, 1, 3, 3, 4], 2),
    ([9, 77, 63, 22, 92, 9, 14, 54, 8, 38, 18, 19, 38, 68, 58, 19], 5),
    ([5] * 50 + list(range(100, 150)), 1),    # một giá trị chiếm đúng nửa
]
for arr, want in cases:
    for name, f in solvers.items():
        if name == "brute" and len(set(arr)) > 18:
            continue
        assert f(arr) == want, (name, arr, f(arr), want)
print("Tất cả", len(cases), "case qua cho", len(solvers), "lời giải")

Tất cả 8 case qua cho 8 lời giải


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [20]:
rng = random.Random(1338)
for _ in range(1500):
    n = 2 * rng.randint(1, 12)
    hi = rng.choice([2, 4, 8, 30])
    arr = [rng.randint(1, hi) for _ in range(n)]
    want = min_set_size_brute(arr)
    for name, f in solvers.items():
        assert f(arr) == want, (name, arr)
print("1500 mảng ngẫu nhiên: mọi cách khớp brute force")

1500 mảng ngẫu nhiên: mọi cách khớp brute force


### 12.3 Kích thước lớn nhất `n = 10^5`

In [21]:
rng = random.Random(5)
big_cases = {
    "ngẫu nhiên 1..10^5": [rng.randint(1, 10**5) for _ in range(10**5)],
    "toàn khác nhau": list(range(1, 10**5 + 1)),
    "lệch (Zipf-ish)": [min(10**5, int(1 / (rng.random() + 1e-9))) for _ in range(10**5)],
}
for label, arr in big_cases.items():
    res = {}
    for name in ["sort", "heap", "bucket", "submit_bucket"]:
        t0 = time.time()
        res[name] = solvers[name](arr)
        res[name + "_t"] = (time.time() - t0) * 1000
    assert len({res[k] for k in ["sort", "heap", "bucket", "submit_bucket"]}) == 1
    print(f"{label:20}: đáp án {res['sort']:6} | " + ", ".join(f"{k} {res[k + '_t']:.0f}ms" for k in ["sort", "heap", "bucket", "submit_bucket"]))

ngẫu nhiên 1..10^5  : đáp án  19797 | sort 15ms, heap 15ms, bucket 12ms, submit_bucket 11ms
toàn khác nhau      : đáp án  50000 | sort 11ms, heap 18ms, bucket 10ms, submit_bucket 9ms


lệch (Zipf-ish)     : đáp án      2 | sort 15ms, heap 5ms, bucket 5ms, submit_bucket 7ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `>` thay vì `>=` (đòi xoá **hơn** một nửa)

In [22]:
def wrong_strict(arr):
    n = len(arr)
    removed = 0
    for ans, f in enumerate(sorted(Counter(arr).values(), reverse=True), 1):
        removed += f
        if 2 * removed > n:                  # SAI
            return ans
    return len(set(arr))

for arr in ([1, 1, 2, 2], examples[0][0]):
    print(arr, "| sai:", wrong_strict(arr), "| đúng:", Solution().minSetSize(arr))

[1, 1, 2, 2] | sai: 2 | đúng: 1
[3, 3, 3, 3, 5, 5, 5, 2, 2, 7] | sai: 2 | đúng: 2


**Vì sao sai:** đề nói "**ít nhất** một nửa" ⇒ xoá đúng một nửa đã đủ. Với `[1,1,2,2]`, xoá `{1}` còn 2 phần tử = một nửa ⇒ đáp án `1`, code sai trả `2`. **Sửa:** `2 * removed >= n`.

### Lỗi 2 — Sắp tần suất **tăng** dần

In [23]:
def wrong_ascending(arr):
    n = len(arr)
    removed = 0
    for ans, f in enumerate(sorted(Counter(arr).values()), 1):     # SAI: nhỏ trước
        removed += f
        if 2 * removed >= n:
            return ans

arr = examples[0][0]
print("sai:", wrong_ascending(arr), "| đúng:", Solution().minSetSize(arr))

sai: 3 | đúng: 2


**Vì sao sai:** lấy tần suất nhỏ trước (`1, 2, 3`) phải tới 3 giá trị mới đủ 5 phần tử; tham lam đúng là lấy **lớn trước**. **Sửa:** `reverse=True`.

### Lỗi 3 — Sắp theo **giá trị** thay vì tần suất

In [24]:
def wrong_by_value(arr):
    n = len(arr)
    cnt = Counter(arr)
    removed = 0
    for ans, v in enumerate(sorted(cnt, reverse=True), 1):          # SAI: giá trị lớn trước
        removed += cnt[v]
        if 2 * removed >= n:
            return ans

arr = examples[0][0]
print("sai:", wrong_by_value(arr), "| đúng:", Solution().minSetSize(arr))

sai: 3 | đúng: 2


**Vì sao sai:** giá trị lớn (`7`, `5`, …) không liên quan đến việc xoá được nhiều phần tử. Thứ tự đúng phải dựa trên **tần suất**. **Sửa:** sắp `cnt.values()` (hoặc `sorted(cnt, key=cnt.get, reverse=True)`).

### Lỗi 4 — Đếm bằng `arr.count(x)` cho từng giá trị (quá chậm)

In [25]:
def slow_count(arr):
    n = len(arr)
    freqs = sorted((arr.count(x) for x in set(arr)), reverse=True)   # O(n · d)
    removed = 0
    for ans, f in enumerate(freqs, 1):
        removed += f
        if 2 * removed >= n:
            return ans

arr = list(range(1, 4001)) * 2                # n = 8000, d = 4000
t0 = time.time(); r1 = slow_count(arr); t_slow = time.time() - t0
t0 = time.time(); r2 = Solution().minSetSize(arr); t_fast = time.time() - t0
print(f"cùng kết quả {r1 == r2} | arr.count: {t_slow*1000:.0f} ms | Counter: {t_fast*1000:.1f} ms (n chỉ mới 8000)")

cùng kết quả True | arr.count: 364 ms | Counter: 0.6 ms (n chỉ mới 8000)


**Vì sao sai:** mỗi `arr.count(x)` duyệt cả mảng `O(n)`, gọi cho `d` giá trị ⇒ `O(n·d)`; với `n = d ≈ 10^5` là `10^10` phép — chắc chắn TLE dù kết quả đúng. **Sửa:** đếm **một lượt** bằng `Counter` hoặc mảng đếm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn **giá trị** và xoá **mọi lần xuất hiện**" ⇒ chỉ còn **tần suất** là quan trọng ⇒ `Counter`.
- "Số lượng ít nhất để đạt tổng ≥ X" với các phần tử độc lập ⇒ **tham lam lấy lớn nhất trước** (sort giảm / heap).
- Giá trị cần sắp là số nguyên trong `[1, n]` ⇒ **bucket/counting sort** cho `O(n)`.
- "Ít nhất một nửa" ⇒ `2*x >= n` (số nguyên, có dấu bằng).
- Nghi ngờ tham lam ⇒ brute force tập con với `d` nhỏ để đối chiếu.

## 15. Biến thể của bài

1. **Xoá ít nhất một tỉ lệ `p/q`** của mảng: điều kiện `q·removed >= p·n` (Bài tập 5 mở rộng).
2. **Trả về chính tập giá trị** (không chỉ kích thước).
3. **LeetCode 1481 — Least Number of Unique Integers after K Removals:** ngược lại — xoá đúng `k` **phần tử** để còn ít giá trị phân biệt nhất ⇒ tham lam xoá tần suất **nhỏ** trước.
4. **LeetCode 347 — Top K Frequent Elements:** cùng công cụ Counter + heap/bucket.
5. **LeetCode 1648 / 2279:** tham lam "lấy cái có lợi nhất trước" với nhiều dạng ràng buộc.

In [26]:
def least_unique_after_k_removals(arr, k):
    """LeetCode 1481: xoá đúng k phần tử, tối thiểu hoá số giá trị phân biệt còn lại."""
    freqs = sorted(Counter(arr).values())
    remaining = len(freqs)
    for f in freqs:
        if k >= f:
            k -= f
            remaining -= 1
        else:
            break
    return remaining

def top_k_frequent(nums, k):
    """LeetCode 347 bằng bucket."""
    n = len(nums)
    buckets = [[] for _ in range(n + 1)]
    for v, f in Counter(nums).items():
        buckets[f].append(v)
    res = []
    for f in range(n, 0, -1):
        for v in buckets[f]:
            res.append(v)
            if len(res) == k:
                return res
    return res

print(least_unique_after_k_removals([4, 3, 1, 1, 3, 3, 2], 3), top_k_frequent([1, 1, 1, 2, 2, 3], 2))

2 [1, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập con | tăng dần kích thước, thử mọi tổ hợp | `O(2^d·d)` | `O(d)` | kiểm chứng `d ≤ 15` |
| 2. Counter + sort | tần suất giảm dần, cộng dồn | `O(n + d log d)` | `O(d)` | mặc định, ngắn nhất |
| 3. Heap | heapify, pop lớn nhất đến khi đủ | `O(n + d + k log d)` | `O(d)` | đáp án `k` nhỏ so với `d` |
| 4. Bucket tần suất | đếm "số giá trị có tần suất f" | `O(n + M)` | `O(n + M)` | yêu cầu tuyến tính |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`minSetSize([1,9])` và `minSetSize([1,2,3,4,5,6])` trả về gì?

### Bài 2 (tính tay)
`arr = [9,77,63,22,92,9,14,54,8,38,18,19,38,68,58,19]`. Đếm tần suất, sắp giảm dần, tìm đáp án.

### Bài 3 (điền TODO)
Hoàn thiện `min_set_todo` theo Cách 2.

### Bài 4 (tự viết lại)
Viết lại Cách 4 (bucket theo tần suất) mà không nhìn code: `min_set_bucket_ex(arr)`.

### Bài 5 (mini challenge)
Viết `choose_set(arr, p, q)` trả về **một tập giá trị** có kích thước nhỏ nhất sao cho số phần tử bị xoá `≥ p/q · n` (với `0 < p ≤ q`). Viết hàm kiểm tra và đối chiếu kích thước với brute force.

In [27]:
# Bài 3, 4, 5 — khung (chỉ định nghĩa, không gọi)
def min_set_todo(arr):
    n = len(arr)
    freqs = None          # TODO: tần suất sắp giảm dần
    removed = 0
    # TODO: cộng dồn đến khi 2 * removed >= n
    pass

def min_set_bucket_ex_todo(arr):
    # TODO
    pass

def choose_set_todo(arr, p, q):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) `[1,9]`: `n = 2`, cần xoá 1. (2) Mỗi giá trị xoá 1 phần tử. (3) `[1..6]`: mỗi giá trị xoá 1, cần 3 ⇒ 3.

**Bài 2:** (1) `9, 38, 19` xuất hiện 2 lần, các số khác 1 lần. (2) `n = 16`, cần 8. (3) `2+2+2 = 6`, thêm `1+1` ⇒ 5 giá trị.

**Bài 3:** (1) `sorted(Counter(arr).values(), reverse=True)`. (2) `enumerate(..., 1)`. (3) Trả `ans` khi `2 * removed >= n`.

**Bài 4:** (1) `bucket[f]` = số giá trị có tần suất `f`. (2) `need = (n + 1) // 2`. (3) Mỗi mức lấy `min(bucket[f], ceil(need / f))`.

**Bài 5:** (1) Điều kiện: `q * removed >= p * n`. (2) Sắp cặp `(giá trị, tần suất)` theo tần suất giảm. (3) Thêm giá trị vào tập đến khi đủ; kiểm tra bằng cách xoá thật và đếm.

## 19. Đáp án tham khảo

In [28]:
# Bài 1, 2
print("Bài 1:", Solution().minSetSize([1, 9]), Solution().minSetSize([1, 2, 3, 4, 5, 6]))
assert Solution().minSetSize([1, 9]) == 1 and Solution().minSetSize([1, 2, 3, 4, 5, 6]) == 3
a2 = [9, 77, 63, 22, 92, 9, 14, 54, 8, 38, 18, 19, 38, 68, 58, 19]
print("Bài 2: tần suất giảm dần =", sorted(Counter(a2).values(), reverse=True), "-> đáp án", Solution().minSetSize(a2))
assert Solution().minSetSize(a2) == 5

# Bài 3
def min_set_sol(arr):
    n = len(arr)
    freqs = sorted(Counter(arr).values(), reverse=True)
    removed = 0
    for ans, f in enumerate(freqs, 1):
        removed += f
        if 2 * removed >= n:
            return ans

# Bài 4
def min_set_bucket_ex(arr):
    n = len(arr)
    bucket = [0] * (n + 1)
    for f in Counter(arr).values():
        bucket[f] += 1
    need, ans = (n + 1) // 2, 0
    for f in range(n, 0, -1):
        if bucket[f]:
            take = min(bucket[f], -(-need // f))
            ans += take
            need -= take * f
            if need <= 0:
                break
    return ans

for arr, want in cases:
    assert min_set_sol(arr) == want and min_set_bucket_ex(arr) == want
print("Bài 1–4: qua")

Bài 1: 1 3
Bài 2: tần suất giảm dần = [2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] -> đáp án 5
Bài 1–4: qua


In [29]:
# Bài 5
def choose_set(arr, p, q):
    n = len(arr)
    chosen, removed = set(), 0
    for v, f in sorted(Counter(arr).items(), key=lambda x: -x[1]):
        if q * removed >= p * n:
            break
        chosen.add(v)
        removed += f
    return chosen

def check_choice(arr, p, q, chosen):
    removed = sum(1 for x in arr if x in chosen)
    return q * removed >= p * len(arr)

def min_size_fraction_brute(arr, p, q):
    cnt = Counter(arr)
    items = list(cnt)
    for k in range(0, len(items) + 1):
        for sub in combinations(items, k):
            if q * sum(cnt[x] for x in sub) >= p * len(arr):
                return k

print("Bài 5:", choose_set(examples[0][0], 1, 2), choose_set(examples[0][0], 9, 10))
rng = random.Random(55)
for _ in range(500):
    arr = [rng.randint(1, 6) for _ in range(rng.randint(1, 14))]
    q = rng.randint(1, 5)
    p = rng.randint(1, q)
    s = choose_set(arr, p, q)
    assert check_choice(arr, p, q, s) and len(s) == min_size_fraction_brute(arr, p, q)
print("Bài 5: 500 bộ ngẫu nhiên — tập hợp lệ và kích thước tối thiểu")

Bài 5: {3, 5} {2, 3, 5}
Bài 5: 500 bộ ngẫu nhiên — tập hợp lệ và kích thước tối thiểu


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tham lam theo tần suất:**

```python
from collections import Counter
n, removed = len(arr), 0
for ans, f in enumerate(sorted(Counter(arr).values(), reverse=True), 1):
    removed += f
    if 2 * removed >= n:
        return ans
```

**Template bucket `O(n)`:** `bucket[f] += 1` cho mỗi tần suất; duyệt `f = n..1`, lấy `min(bucket[f], ceil(need/f))`.

**Checklist trước khi submit:**
- [ ] Đếm tần suất **một lượt** (`Counter`/mảng đếm).
- [ ] Sắp tần suất **giảm dần** (không phải giá trị).
- [ ] Điều kiện dừng `2 * removed >= n` (có dấu bằng).
- [ ] Thử `[1,1,2,2]` (đúng một nửa) và mảng toàn khác nhau.
- [ ] Đúng chữ ký: `minSetSize(self, arr: list[int]) -> int`.